In [1]:
# Analysis C: occupancy vs. output (Day 5, roadmap Part 3).
#
# The condition problem: GSE63803's ChIP-seq was run in spg-7(RNAi) worms, but
# Soo's regulon ranks come from nuo-6 and atfs-1(et15/et17) - a different set of
# interventions. Comparing binding calls to induction ranks directly is a
# cross-condition mismatch. The primary analysis here is therefore the
# condition-matched one: Nargund 2012's real spg-7 ATFS-1-dependent gene set (Table
# S3) against these same spg-7-condition ChIP peaks. The cross-condition version
# (peaks vs. the 61-gene regulon, ranked under different interventions) is reported
# second and labelled as such.
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import gzip
import pandas as pd

GFF = "data/raw/c_elegans.PRJNA13758.WS285.annotations.gff3.gz"

# Real gene models only (source == WormBase, feature == gene) - the file also
# carries genetic-map placeholders under the same "gene" feature label
# (interpolated_pmap_position etc.) with no real coordinates; those are excluded by
# the source filter.
genes = {}
with gzip.open(GFF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 9 or f[1] != "WormBase" or f[2] != "gene":
            continue
        attrs = dict(kv.split("=", 1) for kv in f[8].split(";") if "=" in kv)
        gid_raw = attrs.get("ID", "")
        gid = gid_raw.replace("Gene:", "") if gid_raw.startswith("Gene:") else gid_raw
        if not gid.startswith("WBGene"):
            continue
        # the public gene symbol is in "locus=", not "Name=" (Name is always the
        # WBGene ID in this release) - checked directly against the raw file before
        # relying on it.
        genes[gid] = {
            "chr": f[0], "start": int(f[3]), "end": int(f[4]), "strand": f[6],
            "seqname": attrs.get("sequence_name", ""),
            "public_name": attrs.get("locus", ""),
        }

for g in genes.values():
    g["tss"] = g["start"] if g["strand"] == "+" else g["end"]

print(f"Real gene models parsed: {len(genes)}")


Real gene models parsed: 47649


In [2]:
# Operon membership, from the GFF3's explicit genes= attribute (excludes
# deprecated_operon entries). A downstream operon member is trans-spliced off the
# operon head and has no independent promoter of its own.
operons = {}
with gzip.open(GFF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 9 or f[1] != "operon" or f[2] != "operon":
            continue
        attrs = dict(kv.split("=", 1) for kv in f[8].split(";") if "=" in kv)
        name = attrs.get("Name")
        members = attrs.get("genes", "").split(",") if attrs.get("genes") else []
        if name and members:
            operons[name] = members

gene_to_head = {m: members[0] for members in operons.values() for m in members[1:]}
for gid, g in genes.items():
    g["operon_role"] = "downstream" if gid in gene_to_head else "head_or_independent"
    if g["operon_role"] == "downstream" and gene_to_head[gid] in genes:
        g["head_tss"] = genes[gene_to_head[gid]]["tss"]
        g["head_chr"] = genes[gene_to_head[gid]]["chr"]
    else:
        g["head_tss"] = None

print(f"Operons parsed: {len(operons)}; downstream members: {len(gene_to_head)}")


Operons parsed: 1385; downstream members: 2107


In [3]:
# Peak-to-gene assignment. A gene counts as bound if a peak falls within the window
# of EITHER its own TSS or (for downstream operon members) its operon head's TSS -
# checking both, not only the operon head. An earlier version of this rule checked
# only the operon head for downstream genes and produced a real error: tspo-1 has a
# peak sitting almost exactly on its own gene body (confirmed against the peaks'
# original MACS gene-name annotation, "C41G7.9" = tspo-1), but that peak fell just
# outside the window from the operon head alone (2,150bp vs. the 2kb cutoff), which
# silently flipped a real "bound" to an incorrect "not bound." Checking both
# resolved it without weakening the operon logic that correctly helps elsewhere.
peaks = pd.read_csv("data/liftover/peaks_ce11.bed", sep="\t", header=None,
                     names=["chr", "start", "end", "name"])
peaks["chr"] = peaks["chr"].str.replace("^chr", "", regex=True)

def bound_genes_at_window(window):
    by_chr_own = {}
    by_chr_head = {}
    for gid, g in genes.items():
        by_chr_own.setdefault(g["chr"], []).append((gid, g["tss"]))
        if g["operon_role"] == "downstream" and g["head_tss"] is not None:
            by_chr_head.setdefault(g["head_chr"], []).append((gid, g["head_tss"]))
    bound = set()
    for _, p in peaks.iterrows():
        for gid, tss in by_chr_own.get(p["chr"], []):
            if (p["start"] - window) <= tss <= (p["end"] + window):
                bound.add(gid)
        for gid, tss in by_chr_head.get(p["chr"], []):
            if (p["start"] - window) <= tss <= (p["end"] + window):
                bound.add(gid)
    return bound

bound_2kb = bound_genes_at_window(2000)
print(f"Peaks: {len(peaks)}")
print(f"Genes called bound genome-wide (2kb, own-or-head TSS): {len(bound_2kb)}")

# Validate against the 4 chaperone/QC genes with independently reconciled ground
# truth (gate_decisions.md, "Binding reconciliation"): hsp-6 bound, hsp-60 bound,
# dnj-10 not bound, ymel-1 bound.
name_to_gid = {g["public_name"].lower(): gid for gid, g in genes.items() if g.get("public_name")}
expected = {"hsp-6": True, "hsp-60": True, "dnj-10": False, "ymel-1": True}
print("\n--- Validation ---")
ok = True
for name, exp in expected.items():
    gid = name_to_gid.get(name)
    got = gid in bound_2kb
    if got != exp:
        ok = False
    print(f"  {name}: expected {exp}, got {got}  [{'OK' if got==exp else 'MISMATCH'}]")
if not ok:
    raise RuntimeError("Peak assignment failed validation - do not trust downstream results.")
print("\nAll validations passed.")


Peaks: 1005
Genes called bound genome-wide (2kb, own-or-head TSS): 3003

--- Validation ---
  hsp-6: expected True, got True  [OK]
  hsp-60: expected True, got True  [OK]
  dnj-10: expected False, got False  [OK]
  ymel-1: expected True, got True  [OK]

All validations passed.


In [4]:
# Primary, condition-matched comparison: of Nargund 2012's real ATFS-1-dependent
# spg-7(RNAi) gene set (Table S3), how many are ATFS-1-bound?
table_s3 = pd.read_excel(
    "data/raw/nargund2012_TableS3_spg7_ATFS1dependent.xlsx", sheet_name="Sheet1", header=None
)
table_s3.columns = ["seqname","symbol","function","wt_fold","atfs1_fold","fold_diff","blank","pct_less","na"]

# A real gene row has a numeric wt_fold value. dropna(subset=["symbol"]) silently
# drops every gene with no assigned public symbol (229 of 391 real rows here,
# including the paper's own anchor genes C07G1.7 and F22B3.7) and wrongly counts the
# literal column-header row ("Sequence Name"/"Gene symbol") as a gene, since "Gene
# symbol" is a non-null string. Both bugs are fixed here; see gate_decisions.md,
# "Table S2/S3 row count correction."
table_s3["wt_fold_numeric"] = pd.to_numeric(table_s3["wt_fold"], errors="coerce")
table_s3 = table_s3[table_s3["wt_fold_numeric"].notna()].copy()
table_s3["seqname"] = table_s3["seqname"].astype(str).str.strip()
table_s3["symbol"] = table_s3["symbol"].astype(str).str.strip()

seq_to_gid = {g["seqname"].lower(): gid for gid, g in genes.items() if g.get("seqname")}
table_s3["gid"] = table_s3["seqname"].str.lower().map(seq_to_gid)
unresolved = table_s3["gid"].isna().sum()
table_s3["bound"] = table_s3["gid"].isin(bound_2kb)

n_bound = table_s3["bound"].sum()
n_total = len(table_s3)
print(f"Table S3 genes: {n_total} ({unresolved} unresolved to a current gene ID)")
print(f"\n=== PRIMARY (condition-matched): {n_bound} of {n_total} spg-7 ATFS-1-dependent genes are ATFS-1-bound ===")
print(f"({100*n_bound/n_total:.1f}%)")


Table S3 genes: 391 (10 unresolved to a current gene ID)

=== PRIMARY (condition-matched): 101 of 391 spg-7 ATFS-1-dependent genes are ATFS-1-bound ===
(25.8%)


/opt/homebrew/Caskroom/miniforge/base/envs/atfs1/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [5]:
# Secondary, cross-condition comparison: the 61-gene regulon (ranked under nuo-6 /
# atfs-1(et15,et17), a different condition than the ChIP) against the same binding
# calls, and against Soo's own published binding column for comparison.
soo_df = pd.read_excel("data/raw/ATFS1_targets_Soo.xlsx", sheet_name="Sheet1")
soo_df = soo_df.iloc[0:64].dropna(subset=["Gene name"])
soo_df = soo_df.rename(columns={
    "ATFS-1 bound \nin ChIP-seq": "soo_bound",
    "Gene sequence \nname": "seqname",
})[["Gene name", "seqname", "Score", "Score/variability", "soo_bound"]]
regulon_df = soo_df[~soo_df["Gene name"].isin(["hsp-6", "hsp-60"])].reset_index(drop=True)

seq_to_gid_r = {g["seqname"].lower(): gid for gid, g in genes.items() if g.get("seqname")}
regulon_df["gid"] = regulon_df["seqname"].str.lower().map(seq_to_gid_r)
regulon_df["my_bound"] = regulon_df["gid"].isin(bound_2kb)
regulon_df["soo_bound_bool"] = regulon_df["soo_bound"].str.strip().str.lower() == "yes"

print(f"=== SECONDARY (cross-condition): {regulon_df['my_bound'].sum()} of 61 regulon genes are ATFS-1-bound ===")
print(f"(Soo's own column states {regulon_df['soo_bound_bool'].sum()} of 61 -"
      f" the roadmap's cited anchor number, 22/61, matches Soo's column exactly)")

disagree = regulon_df[regulon_df["soo_bound_bool"] != regulon_df["my_bound"]]
print(f"\nDisagreements with Soo's column: {len(disagree)}, all in the same direction")
print("(this pipeline finds additional bound genes; never fails to find one Soo calls bound):")
print(disagree[["Gene name","soo_bound_bool","my_bound"]].to_string(index=False))


=== SECONDARY (cross-condition): 29 of 61 regulon genes are ATFS-1-bound ===
(Soo's own column states 22 of 61 - the roadmap's cited anchor number, 22/61, matches Soo's column exactly)

Disagreements with Soo's column: 7, all in the same direction
(this pipeline finds additional bound genes; never fails to find one Soo calls bound):
Gene name  soo_bound_bool  my_bound
    srm-3           False      True
  nhr-115           False      True
    DC2.5           False      True
 F56C11.3           False      True
 clec-265           False      True
  M01F1.4           False      True
   ymel-1           False      True


In [6]:
# The 2x2: is the chaperone/protease census fraction higher among bound genes than
# among the regulon overall? Uses this pipeline's own binding calls (validated
# above), on the condition-matched primary set (Table S3, corrected row count - see cell above).
census_df = pd.read_csv("data/chaperone_protease_census.csv")
census_keys = set(census_df["public_name"].str.lower()) | set(census_df["seqname"].str.lower())

table_s3["is_census"] = table_s3["symbol"].str.lower().isin(census_keys) | table_s3["seqname"].str.lower().isin(census_keys)

from scipy.stats import fisher_exact
contingency = pd.crosstab(table_s3["bound"], table_s3["is_census"])
print(f"Contingency table (rows=bound, cols=is_census), Table S3 (n={len(table_s3)}):")
print(contingency)

odds, p = fisher_exact(contingency)
print(f"\nFisher's exact test: odds ratio={odds:.2f}, p={p:.4f}")
print(f"Census genes among bound: {table_s3[table_s3['bound']]['is_census'].sum()} of {table_s3['bound'].sum()}")
print(f"Census genes among unbound: {table_s3[~table_s3['bound']]['is_census'].sum()} of {(~table_s3['bound']).sum()}")


Contingency table (rows=bound, cols=is_census), Table S3 (n=391):
is_census  False  True 
bound                  
False        289      1
True         100      1

Fisher's exact test: odds ratio=2.89, p=0.4504
Census genes among bound: 1 of 101
Census genes among unbound: 1 of 290
